# Data Cleaning

### Objective
Demonstrate professional-level data cleaning by transforming a messy dataset into a clean, analysis-ready dataset using Python, pandas, and Numpy.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display all columns
pd.set_option('display.max_columns', None)

In [17]:
df = pd.read_csv("dirty_cafe_sales.csv")


In [18]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,9/8/2023
1,TXN_4977031,Cake,4,3,12,Cash,In-store,5/16/2023
2,TXN_4271903,Cookie,4,1,ERROR,Credit Card,In-store,7/19/2023
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,4/27/2023
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,6/11/2023


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [20]:
print("Dataset Shape:", df.shape)

Dataset Shape: (10000, 8)


# Data Quality Report

This section checks the overall quality of the dataset before cleaning.

The following will be examined:

- Missing values
- Duplicate rows
- Data types
- Value range anomalies
- Overall dataset structure

In [21]:
# Missing values
print("Misssing Values")
print(df.isnull().sum())

print("\n")

# Duplicate rows
print("Duplicate Rows:", df.duplicated().sum())

print("\n")

# Data types
print(df.dtypes)

print("\n")

# Statistical Summary
print(df.describe(include="all"))

Misssing Values
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


Duplicate Rows: 0


Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object


       Transaction ID   Item Quantity Price Per Unit Total Spent  \
count           10000   9667     9862           9821        9827   
unique          10000     10        7              8          19   
top       TXN_1961373  Juice        5              3           6   
freq                1   1171     2013           2429         979   

        Payment Method  Location Transaction Date  
count             7421      6735             9841  
unique               5         4              367  
top     Digital Wallet  Takeaway          UN

# Missing Value Handling

This dataset contains missing values across several columns.

Cleaning strategy:

- **Items:** Replace missing values with the most frequent item (Mode).
- **Quantity:** Convert to numeric and replace missing values with the median.
- **Price Per Unit:** Convert to numeric and replace missing values with the median.
- **Total Spent:** Convert to numeric and recalculate where possible or replace missing values with the median.
- **Payment Method:** Replace missing values with the mode.
- **Location:** Replace missing values with the mode.
- **Transaction Date:** Convert to datetime and remove rows with invalid or missing dates.

These strategies preserve as much data as possible while minimizing bias.


In [22]:
# Replace invalid values with NaN

df.replace(["ERROR", "UNKNOWN," ""], np.nan, inplace=True)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,9/8/2023
1,TXN_4977031,Cake,4,3,12,Cash,In-store,5/16/2023
2,TXN_4271903,Cookie,4,1,NaN,Credit Card,In-store,7/19/2023
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,4/27/2023
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,6/11/2023
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2,4,NaN,UNKNOWN,8/30/2023
9996,TXN_9659401,NaN,3,NaN,3,Digital Wallet,NaN,6/2/2023
9997,TXN_5255387,Coffee,4,2,8,Digital Wallet,NaN,3/2/2023
9998,TXN_7695629,Cookie,3,NaN,3,Digital Wallet,NaN,12/2/2023


In [23]:
df.isnull().sum()

Transaction ID         0
Item                 625
Quantity             308
Price Per Unit       369
Total Spent          337
Payment Method      2885
Location            3623
Transaction Date     301
dtype: int64

# Data Type Correction

The dataset stores all columns as strings. Before handling missing values, the numeric and date columns are converted to their appropriate data types.

- Quantity - Numeric
- Price Per Unit - Numeric
- Total Spent - Numeric
- Transaction Date - Datetime 

In [24]:
# Convert numeric columns

numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Convert Transaction Date

df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

# Check the new data types

df.dtypes

Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

In [25]:
# Fill categorical columns with mode
df["Item"] = df["Item"].fillna(df["Item"].mode()[0])
df["Payment Method"] = df["Payment Method"].fillna(df["Payment Method"].mode()[0])
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

# Fill numeric columns with median
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Price Per Unit"].median())
df["Total Spent"] = df["Total Spent"].fillna(df["Total Spent"].median())

# Fill date with forward fill
df["Transaction Date"] = df["Transaction Date"].ffill()

## Duplicate Removal

Duplicate records can affect the accuracy of analysis by counting the same transaction multiple times.

The dataset was checked for duplicate rows. No duplicate records were found, so no rows were removed.

In [26]:
duplicates_before = df.duplicated().sum()

print ("Duplicate Rows Before:", duplicates_before)

df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()

print("Duplicate Rows After:", duplicates_after)

Duplicate Rows Before: 0
Duplicate Rows After: 0


## Data Standardisation

The dataset contained inconsistent text formatting and placeholder values such as "UNKNOWN".

Text values were standardised by removing extra spaces and applying title case to categorical columns. This ensures consistency and improves data quality.


In [27]:
# Remove extra spaces
text_columns = ["Item", "Payment Method", "Location"]

for col in text_columns:
    df[col] = df[col].str.strip()

# Standardize text
df["Item"] = df["Item"].str.title()
df["Payment Method"] = df["Payment Method"].str.title()
df["Location"] = df["Location"].str.title()

df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,8.0,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11


## Outlier Detection

The Interquartile Range (IQR) method was used to detect outliers in the numeric columns.

Outliers were identified but retained because they may represent genuine customer purchases rather than data entry errors.

In [28]:
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}: {len(outliers)} outliers")

Quantity: 0 outliers
Price Per Unit: 0 outliers
Total Spent: 259 outliers


## Before vs After Summary

The table below compares the dataset before and after cleaning.

It summarises the improvement 

In [29]:
summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Duplicate Rows",
        "Missing Values",
        "Correct Data Types"
    ],
    "Before": [
        10000,
        duplicates_before,
        "High",
        "No"
    ],
    "After": [
        len(df),
        duplicates_after,
        int(df.isnull().sum().sum()),
        "Yes"
    ]

})

summary

,Metric,Before,After
0,Rows,10000,10000
1,Duplicate Rows,0,0
2,Missing Values,High,0
3,Correct Data Types,No,Yes


## Save Clean Dataset

The cleaned dataset was exported to a new CSV file for future analysis and modelling.

In [30]:
df.to_csv("cleaned_cafe_sales.csv", index=False)

print("Dataset saved successfully.")

Dataset saved successfully.


# Conclusion

The dataset was successfully cleaned and transformed into an analysis-ready format.

Cleaning steps completed include:

- Identified missing values
- Replaced invalid entries with NaN
- Imputed missing values using appropriate statistical methods
- Standardized inconsistent categorical values
- Converted numeric and data columns to correct data types
- Detected and handled outliers usig the IQR method
- Produced a before vs after data quality summary
- Exported the cleaned dataset as cleaned_cafe_sales.csv

The cleaned dataset is now suitable for exploratory data analysis and machine learning.
